# Reading v3 LUCiD Dataset Batches

This notebook demonstrates how to read the v3 four-file dataset layout
(`sensor/`, `inst/`, `seg/`, `labl/` subdirectories under a dataset root
containing `wc_*_NNNN.h5` files per batch).

The v3 readers live in `lucid.sources.event_io`; the convenience helpers
used here live in `lucid.production.data_prod_utils`. See
`docs/LUCID_DATASET.md` for the full schema.

In [ ]:
import sys
sys.path.append('../../../')

from lucid.geometry import generate_detector
from lucid.utils import load_single_event, save_single_event, generate_random_params, print_particle_params
from lucid.simulation import setup_event_simulator
from lucid.generate import read_photon_data_from_photonsim

import jax
import jax.numpy as jnp
import time

from jax import jit
from pathlib import Path

from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

import numpy as np
from functools import partial
import pickle
from tqdm import tqdm
from jax import grad, jit, vmap, value_and_grad
import uproot
from scipy.interpolate import interp1d

import h5py
import numpy as np
import jax.numpy as jnp
import sys
import os

from lucid.utils import analyze_event_kinematics, print_event_kinematics

## Inspect File Structure

First, let's examine the structure of the new multi-event file:

In [ ]:
def inspect_h5_structure(filename):
    """Inspect the structure of any HDF5 file (generic, not schema-specific)."""
    print(f"Inspecting file: {filename}\n")
    print("="*70)

    with h5py.File(filename, 'r') as f:
        print("Top-level keys:", list(f.keys()))
        print("\nFile attributes:", dict(f.attrs))

        def print_structure(name, obj, indent=0):
            prefix = "  " * indent
            if isinstance(obj, h5py.Dataset):
                print(f"{prefix}{name}: Dataset, shape={obj.shape}, dtype={obj.dtype}")
            elif isinstance(obj, h5py.Group):
                print(f"{prefix}{name}: Group")

        print("\nFull structure:")
        f.visititems(print_structure)

    print("="*70)

# v3 dataset root — update to point at a directory with sensor/, inst/, seg/, labl/.
dataset_root = '/sdf/data/neutrino/cjesus/lucid_datasets/e-_10_1500MeV'
file_index = 0
# inspect_h5_structure(f"{dataset_root}/sensor/wc_sensor_{file_index:04d}.h5")

## Updated Read Function

This function handles both single-event and multi-event file structures:

In [ ]:
# v3 convenience helpers — load events directly from a v3 dataset root.
from lucid.production.data_prod_utils import (
    read_multi_event_file,
    load_event_v3,
    print_event_info,
    get_particle_name,
    get_track_hits,
)

## Usage Examples

### Example 1: Read all events from the file

In [ ]:
# Read all events in batch file_index=0 from the dataset root
all_events = read_multi_event_file(dataset_root, file_index=file_index, verbose=False)

In [ ]:
# Check what we got
print(f"\nRead {len(all_events)} events from the file")
print(f"\nFirst event keys: {list(all_events[0].keys())}")

### Example 2: Read a specific event

In [ ]:
# Read a single event using the v3 helper
event_0 = load_event_v3(dataset_root, event_idx=0, file_index=file_index)
print_event_info(event_0)

### Example 3: Analyze kinematics of an event

In [ ]:
# Analyze kinematics for event 0
print_event_kinematics(event_0, show_details=True)

### Example 4: Access event data

In [ ]:
# Access specific fields of the v3 event dict
print(f"source_event_idx: {event_0['source_event_idx']}")
print(f"n_particles:      {event_0['n_particles']}")
print(f"PDG (per particle): {event_0['PDG']}")
print(f"Q shape (particles × sensors): {event_0['Q'].shape}")
print(f"Sensor-summed PE (Q_tot) first 5: {event_0['Q_tot'][:5]}")
print(f"t0: {event_0['t0']:.2f} ns, overall containment: {event_0['overall_containment']*100:.1f}%")
# Track-level truth is under event_0['labl']['per_track']; segments under event_0['seg'].

### Example 5: Iterate through all events

In [ ]:
# Process all events in the batch
all_events = read_multi_event_file(dataset_root, file_index=file_index, verbose=False)

print(f"Processing {len(all_events)} events...\n")
for i, event in enumerate(all_events):
    n_particles = event['n_particles']
    total_charge = float(event['Q_tot'].sum())
    print(f"Event {i}: {n_particles} particles, total PE = {total_charge:.2f}")

### Example 6: Statistics across all events

In [ ]:
# Aggregate statistics across all events in the batch
all_events = read_multi_event_file(dataset_root, file_index=file_index, verbose=False)

total_particles = sum(ev['n_particles'] for ev in all_events)
total_pe = sum(float(ev['Q_tot'].sum()) for ev in all_events)
all_pdgs = np.concatenate([ev['PDG'] for ev in all_events])

print(f"\n{'='*60}")
print("Statistics Across All Events in Batch")
print(f"{'='*60}")
print(f"Total events:              {len(all_events)}")
print(f"Total particles:           {total_particles}")
print(f"Avg particles per event:   {total_particles / len(all_events):.2f}")
print(f"Total detected PE:         {total_pe:.2f}")
print(f"Avg PE per event:          {total_pe / len(all_events):.2f}")

unique_pdg, counts = np.unique(all_pdgs, return_counts=True)
print("\nPDG distribution across all events:")
for pdg, count in zip(unique_pdg, counts):
    name = get_particle_name(int(pdg)) if int(pdg) != -1 else '?'
    pct = 100 * count / total_particles
    print(f"  {name} (PDG {int(pdg):>5}): {count} ({pct:.1f}%)")

## Inspect PMT-level information

In [ ]:
# Per-PMT inspection for a single event
event_0 = load_event_v3(dataset_root, event_idx=0, file_index=file_index)
print_event_info(event_0)

# The v3 event dict exposes the dense per-particle PE/T matrices as Q and T,
# so you can slice any PMT or particle row directly:
print(f"\nQ shape (particles × sensors): {event_0['Q'].shape}")
print(f"First 10 PMTs of particle 0 (PE): {event_0['Q'][0, :10]}")
print(f"Sensor-summed PE across first 10 PMTs: {event_0['Q_tot'][:10]}")